In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expi
from mpmath import mp
from scipy.special import iv, kv, ive, kve
from sklearn.preprocessing import MinMaxScaler
from scipy.signal import savgol_filter

# Cell 1 — Dataset V1 Parameter Space

## This is the final definition of the 500 physical cases.

In [2]:
np.random.seed(42)

n_cases = 500

parameter_cases = pd.DataFrame({
    "case_id": np.arange(n_cases),

    "k_mD": np.random.uniform(
        20, 200, n_cases
    ),

    "skin": np.random.uniform(
        -2, 5, n_cases
    ),

    "C_bbl_per_psi": np.random.uniform(
        0.0025, 0.020, n_cases
    ),

    "re_ft": np.random.uniform(
        125, 1000, n_cases
    ),

    "boundary_type": np.random.choice(
        ["no_flow", "constant_pressure"],
        size=n_cases
    )
})

parameter_cases.head()

,case_id,k_mD,skin,C_bbl_per_psi,re_ft,boundary_type
0,0,87.417221,2.887132,0.005740,579.196562,constant_pressure
1,1,191.128575,1.752675,0.011983,544.284143,no_flow
2,2,151.758910,0.166693,0.017777,147.436808,no_flow
3,3,127.758527,3.696565,0.015314,423.591849,no_flow
4,4,48.083355,2.793118,0.016615,457.671166,constant_pressure


# Cell 2 — Final Time Grid

In [3]:
n_time = 290

time_dataset = np.logspace(
    -4, 2, n_time
)

print("Number of time points:", len(time_dataset))
print("Time range:",
      time_dataset.min(),
      "to",
      time_dataset.max(),
      "hr")

Number of time points: 290
Time range: 0.0001 to 100.0 hr


# Cell 3 — Physical Constants

In [4]:
phi = 0.20
mu = 1.0
ct = 1.5e-5
rw = 0.25
h = 50.0
q = 500.0
B = 1.2

# Cell 4 — Dimensionless Parameters

In [5]:
def calculate_dimensionless_parameters(
    time_hr,
    k,
    phi,
    mu,
    ct,
    rw,
    h,
    C,
    re
):

    tD = (
        0.0002637 * k * time_hr
        / (phi * mu * ct * rw**2)
    )

    CD = (
        0.8936 * C
        / (phi * ct * h * rw**2)
    )

    reD = re / rw

    return tD, CD, reD

# Cell 5 — Bourdet Derivative

In [6]:
def bourdet_derivative(pressure, time):

    pressure = np.asarray(
        pressure,
        dtype=float
    )

    time = np.asarray(
        time,
        dtype=float
    )

    if len(pressure) != len(time):
        raise ValueError(
            "pressure and time must have the same length"
        )

    if np.any(time <= 0):
        raise ValueError(
            "time values must be positive"
        )

    ln_time = np.log(time)

    derivative = np.full(
        len(pressure),
        np.nan
    )

    for i in range(1, len(pressure) - 1):

        dx_left = (
            ln_time[i]
            - ln_time[i - 1]
        )

        dx_right = (
            ln_time[i + 1]
            - ln_time[i]
        )

        slope_left = (
            pressure[i]
            - pressure[i - 1]
        ) / dx_left

        slope_right = (
            pressure[i + 1]
            - pressure[i]
        ) / dx_right

        derivative[i] = (
            slope_left
            * dx_right
            / (dx_left + dx_right)
            +
            slope_right
            * dx_left
            / (dx_left + dx_right)
        )

    return derivative

# Physical Transient Solver

## Cell 6 — Fast Constant-Pressure Boundary Solution

In [7]:
from scipy.special import iv, kv, ive, kve

def pD_constant_pressure_fast(u, reD):

    u = np.asarray(u, dtype=float)

    x = np.sqrt(u)
    xe = reD * x

    I0_x = iv(0, x)
    K0_x = kv(0, x)

    I1_x = iv(1, x)
    K1_x = kv(1, x)

    scaled_ratio = (
        kve(0, xe)
        / ive(0, xe)
    )

    R = (
        scaled_ratio
        * np.exp(-2.0 * xe)
    )

    numerator = (
        K0_x
        - R * I0_x
    )

    denominator = (
        x
        * (K1_x + R * I1_x)
    )

    return (
        (1.0 / u)
        * numerator
        / denominator
    )


def pWD_constant_pressure_fast(
    u,
    reD,
    CD
):

    pD_bar = (
        pD_constant_pressure_fast(
            u,
            reD
        )
    )

    return (
        1.0
        /
        (
            1.0 / pD_bar
            + u**2 * CD
        )
    )

## Cell 7 — Fast No-Flow Boundary Solution

In [8]:
def pD_no_flow_fast(u, reD):

    u = np.asarray(u, dtype=float)

    x = np.sqrt(u)
    xe = reD * x

    I0_x = iv(0, x)
    K0_x = kv(0, x)

    I1_x = iv(1, x)
    K1_x = kv(1, x)

    scaled_ratio = (
        kve(1, xe)
        / ive(1, xe)
    )

    R = (
        scaled_ratio
        * np.exp(-2.0 * xe)
    )

    numerator = (
        K0_x
        + R * I0_x
    )

    denominator = (
        x
        * (K1_x - R * I1_x)
    )

    return (
        (1.0 / u)
        * numerator
        / denominator
    )


def pWD_no_flow_fast(
    u,
    reD,
    CD
):

    pD_bar = (
        pD_no_flow_fast(
            u,
            reD
        )
    )

    return (
        1.0
        /
        (
            1.0 / pD_bar
            + u**2 * CD
        )
    )

## Cell 8 — Stehfest Weights

In [9]:
def stehfest_weights(N=16):

    if N % 2 != 0:
        raise ValueError(
            "N must be even"
        )

    M = N // 2
    V = []

    for k in range(1, N + 1):

        total = mp.mpf("0")

        j_min = (k + 1) // 2
        j_max = min(k, M)

        for j in range(
            j_min,
            j_max + 1
        ):

            numerator = (
                mp.power(j, M)
                * mp.factorial(2 * j)
            )

            denominator = (
                mp.factorial(M - j)
                * mp.factorial(j)
                * mp.factorial(j - 1)
                * mp.factorial(k - j)
                * mp.factorial(2 * j - k)
            )

            total += (
                numerator
                / denominator
            )

        V.append(
            mp.power(-1, k + M)
            * total
        )

    return V


V16 = stehfest_weights(16)

## Cell 9 — Fast Laplace Inversion

### This includes both boundary conditions and the effective-radius treatment of skin.

In [10]:
def inverse_constant_pressure_fast(
    tD,
    CD,
    skin,
    reD,
    weights=V16
):

    factor = np.exp(
        2.0 * skin
    )

    tD_eff = (
        np.asarray(tD, dtype=float)
        * factor
    )

    CD_eff = CD * factor

    reD_eff = (
        reD * np.exp(skin)
    )

    j = np.arange(
        1,
        len(weights) + 1,
        dtype=float
    )

    V = np.array(
        [float(v) for v in weights]
    )

    ln2 = np.log(2.0)

    u = (
        j[None, :]
        * ln2
        / tD_eff[:, None]
    )

    F = pWD_constant_pressure_fast(
        u,
        reD=reD_eff,
        CD=CD_eff
    )

    return (
        ln2
        / tD_eff
        * np.sum(
            F * V[None, :],
            axis=1
        )
    )


def inverse_no_flow_fast(
    tD,
    CD,
    skin,
    reD,
    weights=V16
):

    factor = np.exp(
        2.0 * skin
    )

    tD_eff = (
        np.asarray(tD, dtype=float)
        * factor
    )

    CD_eff = CD * factor

    reD_eff = (
        reD * np.exp(skin)
    )

    j = np.arange(
        1,
        len(weights) + 1,
        dtype=float
    )

    V = np.array(
        [float(v) for v in weights]
    )

    ln2 = np.log(2.0)

    u = (
        j[None, :]
        * ln2
        / tD_eff[:, None]
    )

    F = pWD_no_flow_fast(
        u,
        reD=reD_eff,
        CD=CD_eff
    )

    return (
        ln2
        / tD_eff
        * np.sum(
            F * V[None, :],
            axis=1
        )
    )

# Generate the Clean Dataset
## Cell 10 — Generate One Physical Transient

In [11]:
def generate_one_case_fast(row):

    tD, CD, reD = (
        calculate_dimensionless_parameters(
            time_hr=time_dataset,
            k=row["k_mD"],
            phi=phi,
            mu=mu,
            ct=ct,
            rw=rw,
            h=h,
            C=row["C_bbl_per_psi"],
            re=row["re_ft"]
        )
    )

    if row["boundary_type"] == "no_flow":

        pWD = inverse_no_flow_fast(
            tD=tD,
            CD=CD,
            skin=row["skin"],
            reD=reD
        )

    elif (
        row["boundary_type"]
        == "constant_pressure"
    ):

        pWD = (
            inverse_constant_pressure_fast(
                tD=tD,
                CD=CD,
                skin=row["skin"],
                reD=reD
            )
        )

    else:
        raise ValueError(
            "Unknown boundary type"
        )

    pressure_factor = (
        141.2
        * q
        * mu
        * B
        /
        (row["k_mD"] * h)
    )

    delta_p = (
        pressure_factor
        * pWD
    )

    derivative = bourdet_derivative(
        delta_p,
        time_dataset
    )

    return {
        "case_id":
            int(row["case_id"]),

        "boundary_type":
            row["boundary_type"],

        "pWD":
            pWD,

        "delta_p_psi":
            delta_p,

        "derivative_psi":
            derivative,

        "tD":
            tD,

        "CD":
            CD,

        "reD":
            reD
    }

## Cell 11 — Generate All 500 Clean Cases

In [12]:
clean_cases = []

for _, row in parameter_cases.iterrows():

    case = generate_one_case_fast(
        row
    )

    case["k_mD"] = float(
        row["k_mD"]
    )

    case["skin"] = float(
        row["skin"]
    )

    case["C_bbl_per_psi"] = float(
        row["C_bbl_per_psi"]
    )

    case["re_ft"] = float(
        row["re_ft"]
    )

    clean_cases.append(case)


print(
    "Generated cases:",
    len(clean_cases)
)

print(
    "Points per case:",
    len(time_dataset)
)

print(
    "Total transient points:",
    len(clean_cases)
    * len(time_dataset)
)

Generated cases: 500
Points per case: 290
Total transient points: 145000


# Freeze the Training/Test Cases

## Cell 12 — Space-Filling 100/400 Split

In [13]:
selection_features = [
    "k_mD",
    "skin",
    "C_bbl_per_psi",
    "re_ft"
]

selection_scaler = MinMaxScaler()

X_selection = (
    selection_scaler.fit_transform(
        parameter_cases[
            selection_features
        ]
    )
)

selection_df = (
    parameter_cases.copy()
)

for i, feature in enumerate(
    selection_features
):
    selection_df[
        f"{feature}_scaled"
    ] = X_selection[:, i]

# Cell 13 — Farthest-Point Selection

In [14]:
def farthest_point_selection(
    X,
    n_select,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    selected = [
        rng.integers(len(X))
    ]

    min_dist = np.full(
        len(X),
        np.inf
    )

    for _ in range(
        1,
        n_select
    ):

        last = X[
            selected[-1]
        ]

        dist = np.linalg.norm(
            X - last,
            axis=1
        )

        min_dist = np.minimum(
            min_dist,
            dist
        )

        next_idx = np.argmax(
            min_dist
        )

        selected.append(
            next_idx
        )

    return selected

In [15]:
scaled_cols = [
    f"{x}_scaled"
    for x in selection_features
]

train_ids = []

for boundary in [
    "no_flow",
    "constant_pressure"
]:

    subset = selection_df[
        selection_df["boundary_type"]
        == boundary
    ].copy()

    X_sub = subset[
        scaled_cols
    ].to_numpy()

    selected_local = (
        farthest_point_selection(
            X_sub,
            n_select=50,
            seed=42
        )
    )

    selected_ids = (
        subset.iloc[selected_local]
        ["case_id"]
        .tolist()
    )

    train_ids.extend(
        selected_ids
    )

train_ids = set(train_ids)

train_params = parameter_cases[
    parameter_cases["case_id"].isin(
        train_ids
    )
].copy()

test_params = parameter_cases[
    ~parameter_cases["case_id"].isin(
        train_ids
    )
].copy()

print(
    "Training cases:",
    len(train_params)
)

print(
    "Testing cases:",
    len(test_params)
)

print("\nTraining boundary counts:")
print(
    train_params[
        "boundary_type"
    ].value_counts()
)

print("\nTesting boundary counts:")
print(
    test_params[
        "boundary_type"
    ].value_counts()
)

Training cases: 100
Testing cases: 400

Training boundary counts:
boundary_type
no_flow              50
constant_pressure    50
Name: count, dtype: int64

Testing boundary counts:
boundary_type
constant_pressure    208
no_flow              192
Name: count, dtype: int64


In [16]:
train_case_ids = sorted(
    train_params["case_id"].tolist()
)

test_case_ids = sorted(
    test_params["case_id"].tolist()
)

print(
    "Training cases:",
    len(train_case_ids)
)

print(
    "Testing cases:",
    len(test_case_ids)
)

Training cases: 100
Testing cases: 400


# Convert Clean Data to Observed Data

## Cell 14 — Add Pressure Noise and Smooth the Transients

In [17]:
sigma_abs = 0.02
relative_fraction = 0.001

observed_cases = []

for case in clean_cases:

    p_clean = case[
        "delta_p_psi"
    ]

    rng = np.random.default_rng(
        1000 + case["case_id"]
    )

    sigma_pressure = np.sqrt(
        sigma_abs**2
        +
        (
            relative_fraction
            * p_clean
        )**2
    )

    p_noisy = (
        p_clean
        + rng.normal(
            0.0,
            sigma_pressure
        )
    )

    p_smooth = savgol_filter(
        p_noisy,
        window_length=21,
        polyorder=2
    )

    d_observed = (
        bourdet_derivative(
            p_smooth,
            time_dataset
        )
    )

    observed_cases.append({
        "case_id":
            case["case_id"],

        "boundary_type":
            case["boundary_type"],

        "k_mD":
            case["k_mD"],

        "skin":
            case["skin"],

        "C_bbl_per_psi":
            case["C_bbl_per_psi"],

        "re_ft":
            case["re_ft"],

        "pressure_clean":
            p_clean.copy(),

        "pressure_noisy":
            p_noisy,

        "pressure_smoothed":
            p_smooth,

        "derivative_clean":
            case[
                "derivative_psi"
            ].copy(),

        "derivative_observed":
            d_observed,

        "noise_sigma":
            sigma_pressure
    })

## Cell 15 — Final Observed-Data QC

In [18]:
print(
    "Cases:",
    len(observed_cases)
)

print(
    "All smoothed pressures finite:",
    all(
        np.all(
            np.isfinite(
                c["pressure_smoothed"]
            )
        )
        for c in observed_cases
    )
)

print(
    "All interior derivatives finite:",
    all(
        np.all(
            np.isfinite(
                c["derivative_observed"][1:-1]
            )
        )
        for c in observed_cases
    )
)

Cases: 500
All smoothed pressures finite: True
All interior derivatives finite: True


# Final Feature Extraction
## Cell 16 — Fixed Feature Times and Log-Time Interpolation

In [19]:
feature_times = np.array([
    0.01,
    0.03,
    0.1,
    0.5,
    1.0,
    3.0,
    10.0,
    30.0
])


def interpolate_log_time(
    time,
    values,
    target_times
):

    return np.interp(
        np.log10(target_times),
        np.log10(time),
        values
    )

## Cell 17 — Build the ML Feature Table

In [20]:
feature_rows = []

for case in observed_cases:

    p_features = (
        interpolate_log_time(
            time_dataset,
            case[
                "pressure_smoothed"
            ],
            feature_times
        )
    )

    d_features = (
        interpolate_log_time(
            time_dataset,
            case[
                "derivative_observed"
            ],
            feature_times
        )
    )

    row = {
        "case_id":
            case["case_id"]
    }

    for t, value in zip(
        feature_times,
        p_features
    ):
        row[
            f"P_{t:g}hr"
        ] = value

    for t, value in zip(
        feature_times,
        d_features
    ):
        row[
            f"D_{t:g}hr"
        ] = value

    row["k_mD"] = (
        case["k_mD"]
    )

    row["skin"] = (
        case["skin"]
    )

    feature_rows.append(row)


feature_df = pd.DataFrame(
    feature_rows
)

print(
    "Feature table shape:",
    feature_df.shape
)

Feature table shape: (500, 19)


## Cell 18 — Final Baseline X and y

In [21]:
feature_columns = [
    col
    for col in feature_df.columns
    if (
        col.startswith("P_")
        or col.startswith("D_")
    )
]

target_columns = [
    "k_mD",
    "skin"
]

X = feature_df[
    feature_columns
].copy()

y = feature_df[
    target_columns
].copy()

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "\nFeatures:"
)

print(
    feature_columns
)

print(
    "\nTargets:"
)

print(
    target_columns
)

X shape: (500, 16)
y shape: (500, 2)

Features:
['P_0.01hr', 'P_0.03hr', 'P_0.1hr', 'P_0.5hr', 'P_1hr', 'P_3hr', 'P_10hr', 'P_30hr', 'D_0.01hr', 'D_0.03hr', 'D_0.1hr', 'D_0.5hr', 'D_1hr', 'D_3hr', 'D_10hr', 'D_30hr']

Targets:
['k_mD', 'skin']


## Cell 19 — Save the ML-ready dataset

In [22]:
ml_dataset = feature_df.copy()

train_id_set = set(train_case_ids)

ml_dataset["split"] = np.where(
    ml_dataset["case_id"].isin(train_id_set),
    "train",
    "test"
)

ml_dataset.to_csv(
    "well_test_ml_dataset.csv",
    index=False
)

print(ml_dataset.shape)
print(ml_dataset["split"].value_counts())

(500, 20)
split
test     400
train    100
Name: count, dtype: int64


## Cell 20 — Save the processed transients

In [23]:
np.savez_compressed(
    "well_test_processed_transients.npz",

    case_id=np.array([
        c["case_id"]
        for c in observed_cases
    ]),

    time_hr=time_dataset,

    pressure_clean=np.array([
        c["pressure_clean"]
        for c in observed_cases
    ]),

    pressure_noisy=np.array([
        c["pressure_noisy"]
        for c in observed_cases
    ]),

    pressure_smoothed=np.array([
        c["pressure_smoothed"]
        for c in observed_cases
    ]),

    derivative_clean=np.array([
        c["derivative_clean"]
        for c in observed_cases
    ]),

    derivative_observed=np.array([
        c["derivative_observed"]
        for c in observed_cases
    ])
)